# 01.2 - Hash Tables & Sets

## What is a hash table?
A structure that stores **key → value** pairs with **O(1) average** insert, lookup and delete.

How: a **hash function** turns the key into a number → `index = hash(key) % number_of_buckets` → store the pair in that bucket.

```
"alice" --hash--> 9472815 % 8 = 7  -> bucket 7: [("alice", 25)]
"bob"   --hash--> 2210349 % 8 = 5  -> bucket 5: [("bob", 31)]
```

**Collision:** two keys land in the same bucket. Fixed by **chaining** (a list per bucket) or **open addressing** (probe the next free slot - what Python's `dict` does).
**Load factor** = items / buckets. When it gets too high the table **resizes** (rehashes everything) to keep operations O(1).

In Python: `dict` = hash map, `set` = hash set (keys only). Keys must be **immutable / hashable** (str, int, tuple - not list).

## Where is it used?
| Real system | Use |
|---|---|
| Python itself | Variables, objects' attributes, modules are all dicts |
| Caches (Redis, Memcached, browser cache) | URL / key → cached response |
| Databases | Hash indexes, hash joins, `GROUP BY` |
| Login systems | username → user record; password **hashes** |
| Deduplication | "Have I seen this transaction / email / file before?" |
| Compilers | Symbol tables (variable name → info) |
| Counting | Word frequencies, analytics, `Counter` |
| Blockchain / Git | Cryptographic hashes identify blocks & commits |

## Complexity
| Operation | Average | Worst (many collisions) |
|---|---|---|
| insert / lookup / delete | **O(1)** | O(n) |

## 1. Build a hash map from scratch (chaining) - also LC #706 Design HashMap

In [1]:
class MyHashMap:
    def __init__(self, capacity=8):
        self.capacity = capacity
        self.size = 0
        self.buckets = [[] for _ in range(capacity)]

    def _index(self, key):
        return hash(key) % self.capacity

    def put(self, key, value):
        bucket = self.buckets[self._index(key)]
        for pair in bucket:
            if pair[0] == key:          # update existing
                pair[1] = value
                return
        bucket.append([key, value])
        self.size += 1
        if self.size / self.capacity > 0.75:   # load factor -> resize
            self._resize()

    def get(self, key, default=-1):
        for k, v in self.buckets[self._index(key)]:
            if k == key:
                return v
        return default

    def remove(self, key):
        bucket = self.buckets[self._index(key)]
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self.size -= 1
                return

    def _resize(self):
        old = [pair for bucket in self.buckets for pair in bucket]
        self.capacity *= 2
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for k, v in old:
            self.put(k, v)

hm = MyHashMap()
for i, name in enumerate(["alice", "bob", "carol", "dave", "eve", "frank", "grace"]):
    hm.put(name, i)
hm.put("bob", 99)
hm.remove("carol")
assert hm.get("bob") == 99 and hm.get("carol") == -1 and hm.get("grace") == 6
print("capacity after resizes:", hm.capacity, "size:", hm.size)
print("MyHashMap ")

capacity after resizes: 16 size: 6
MyHashMap 


In [2]:
# Python's hash() and why lists can't be keys
print(hash("abc"), hash(42), hash((1, 2)))
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("TypeError:", e)

# Set operations
a, b = {1, 2, 3, 4}, {3, 4, 5}
print(a | b, a & b, a - b, a ^ b)

4750671109896449786 42 -3550055125485641917
TypeError: unhashable type: 'list'
{1, 2, 3, 4, 5} {3, 4} {1, 2} {1, 2, 5}


---
## Solved problems

### 1. First Unique Character in a String - LC #387 (Easy)
Return the index of the first non-repeating character, or -1.

**Idea:** Count everything first (one pass), then find the first char with count 1 (second pass).

In [3]:
from collections import Counter

def first_uniq_char(s):
    count = Counter(s)
    for i, ch in enumerate(s):
        if count[ch] == 1:
            return i
    return -1

assert first_uniq_char("leetcode") == 0
assert first_uniq_char("loveleetcode") == 2
assert first_uniq_char("aabb") == -1
print("First Unique Char ")

First Unique Char 


### 2. Intersection of Two Arrays - LC #349 (Easy)
Unique elements present in both arrays.

In [4]:
def intersection(nums1, nums2):
    return list(set(nums1) & set(nums2))

assert sorted(intersection([1, 2, 2, 1], [2, 2])) == [2]
assert sorted(intersection([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9]
print("Intersection ")

Intersection 


### 3. Isomorphic Strings - LC #205 (Easy)
Can characters of `s` be replaced (consistently, one-to-one) to get `t`?

**Idea:** Two dicts to enforce a one-to-one mapping in both directions.

In [5]:
def is_isomorphic(s, t):
    s_to_t, t_to_s = {}, {}
    for a, b in zip(s, t):
        if s_to_t.get(a, b) != b or t_to_s.get(b, a) != a:
            return False
        s_to_t[a] = b
        t_to_s[b] = a
    return True

assert is_isomorphic("egg", "add")
assert not is_isomorphic("foo", "bar")
assert not is_isomorphic("badc", "baba")
print("Isomorphic ")

Isomorphic 


### 4. Ransom Note - LC #383 (Easy)
Can `ransomNote` be built from letters in `magazine` (each letter used once)?

In [6]:
from collections import Counter

def can_construct(ransom_note, magazine):
    need = Counter(ransom_note)
    have = Counter(magazine)
    return all(have[ch] >= cnt for ch, cnt in need.items())
    # or: return not (Counter(ransom_note) - Counter(magazine))

assert not can_construct("aa", "ab")
assert can_construct("aa", "aab")
print("Ransom Note ")

Ransom Note 


### 5. Valid Sudoku - LC #36 (Medium)
Check a 9×9 board: no repeated digit in any row, column or 3×3 box.

**Idea:** One set per row, per column and per box. The box id is `(r // 3, c // 3)` - a tuple used as a dict key.

In [7]:
from collections import defaultdict

def is_valid_sudoku(board):
    rows, cols, boxes = defaultdict(set), defaultdict(set), defaultdict(set)
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            box = (r // 3, c // 3)
            if v in rows[r] or v in cols[c] or v in boxes[box]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[box].add(v)
    return True

board = [["5","3",".",".","7",".",".",".","."],["6",".",".","1","9","5",".",".","."],
         [".","9","8",".",".",".",".","6","."],["8",".",".",".","6",".",".",".","3"],
         ["4",".",".","8",".","3",".",".","1"],["7",".",".",".","2",".",".",".","6"],
         [".","6",".",".",".",".","2","8","."],[".",".",".","4","1","9",".",".","5"],
         [".",".",".",".","8",".",".","7","9"]]
assert is_valid_sudoku(board)
board[0][0] = "8"           # now column 0 has two 8s
assert not is_valid_sudoku(board)
print("Valid Sudoku ")

Valid Sudoku 


## More practice on LeetCode
Easy: Happy Number (#202), Word Pattern (#290), Jewels and Stones (#771)
Medium: Encode and Decode TinyURL (#535), Insert Delete GetRandom O(1) (#380), Longest Consecutive Sequence (#128 - solved in 03.1)

Also in `03-Interview Patterns/03.1`: Two Sum, Contains Duplicate, Valid Anagram, Group Anagrams, Top K Frequent.

## Key takeaways
- dict/set give O(1) average lookups - the #1 tool to turn O(n²) into O(n).
- Keys must be hashable (use tuples, not lists).
- Pattern: *"Have I seen it?"* → set. *"How many times?" / "where?"* → dict / Counter.

**Next: `01.3-Linked-Lists.ipynb`**